# 02 - Total p-Variation Reconstruction for Sparse-View CT

This notebook implements a model-based variational reconstruction method for sparse-view CT. The measured data are the degraded Mayo sinograms generated in notebook 01, and the reconstruction is obtained by minimizing

$$
\hat{x} = \arg\min_x \frac{1}{2}\|Kx-y^\delta\|_2^2 + \lambda \mathcal{R}_{TpV}(x),
$$

where $K$ is the parallel-beam CT projector, $y^\delta$ is the noisy sinogram, and $\mathcal{R}_{TpV}(x)=\sum_i \|\nabla x_i\|_2^p$ is the Total $p$-Variation prior. The value of $p$ is kept in the non-convex sparse range $0.1 < p < 0.5$, which encourages piecewise regular reconstructions while preserving sharp anatomical boundaries.

The method acts directly on sinograms and enforces data fidelity in the Radon domain. FBP is computed only as an initialization for the iterative solver and as a baseline for quantitative and visual comparison.

In [ ]:
!pip install astra-toolbox

from google.colab import drive

drive.mount("/content/drive")

import csv
import json
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import torch

PROJECT_ROOT = Path("/content/drive/MyDrive/LM_INFORMATICA/COMPUTATIONAL_IMAGING")
PROCESSED_DIR = PROJECT_ROOT / "processed"
OUTPUT_DIR = PROJECT_ROOT / "outputs"
RECONSTRUCTION_DIR = OUTPUT_DIR / "reconstructions" / "tpv"
FIGURES_DIR = OUTPUT_DIR / "figures"
METRICS_DIR = OUTPUT_DIR / "metrics"

sys.path.append(str(PROJECT_ROOT))

from IPPy import operators, solvers, utilities
from IPPy.utilities import normalize
from IPPy.utilities import metrics

SEED = 42
IMAGE_SIZE = 256
ANGLE_COUNTS = (180, 90, 60, 45)
DETECTOR_SIZE = 256
GEOMETRY = "parallel"

for directory in [RECONSTRUCTION_DIR, FIGURES_DIR, METRICS_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

torch.manual_seed(SEED)
np.random.seed(SEED)

device = utilities.get_device()
print("Device used:", device)
print("Processed data:", PROCESSED_DIR)
print("Reconstructions:", RECONSTRUCTION_DIR)
print("Figures:", FIGURES_DIR)
print("Metrics:", METRICS_DIR)

## Processed Test-Set Contract

The notebook reads the fixed degraded data produced by notebook 01. Each patient file is a PyTorch dictionary containing clean images, noisy sinograms for the four sparse-view settings, source paths, and metadata. No new noisy measurements are generated here: all methods are evaluated on the same saved sinograms.

In [ ]:
manifest_path = PROCESSED_DIR / "manifest.json"
with manifest_path.open("r", encoding="utf-8") as file:
    manifest = json.load(file)

test_split = manifest["splits"]["test"]
if "patients" not in test_split:
    available_keys = ", ".join(sorted(test_split.keys()))
    raise RuntimeError(
        "The processed manifest does not use the patient-level contract. "
        f"Expected key 'patients', found keys: {available_keys}. "
        "Re-run notebook 01 before executing this notebook."
    )

test_patient_record = test_split["patients"][0]
test_patient_path = PROCESSED_DIR / test_patient_record["path"]
payload = torch.load(test_patient_path, map_location="cpu")

clean = payload["clean"].detach().cpu().to(torch.float32)
sinograms = {key: value.detach().cpu().to(torch.float32) for key, value in payload["sinograms"].items()}
source_paths = payload["source_paths"]
metadata = payload["metadata"]
patient_id = metadata["patient_id"]

expected_angle_keys = [str(n_views) for n_views in ANGLE_COUNTS]
if tuple(clean.shape[1:]) != (1, IMAGE_SIZE, IMAGE_SIZE):
    raise RuntimeError(f"Unexpected clean tensor shape: {tuple(clean.shape)}")

for angle_key, n_views in zip(expected_angle_keys, ANGLE_COUNTS):
    if angle_key not in sinograms:
        raise RuntimeError(f"Missing sinogram for {angle_key} views")
    expected_shape = (clean.shape[0], 1, n_views, DETECTOR_SIZE)
    if tuple(sinograms[angle_key].shape) != expected_shape:
        raise RuntimeError(
            f"Unexpected sinogram shape for {angle_key} views: "
            f"{tuple(sinograms[angle_key].shape)} != {expected_shape}"
        )

print("Patient file:", test_patient_path)
print("Patient ID:", patient_id)
print("Clean:", tuple(clean.shape))
for angle_key in expected_angle_keys:
    print(f"Sinogram {angle_key}:", tuple(sinograms[angle_key].shape))
print("First source path:", source_paths[0])

## Sparse-View CT Geometry and Radon-Domain Fidelity

For each angular configuration, the physical operator is the same parallel-beam CT projector used during degradation. The objective function compares $Kx$ with the saved noisy sinogram $y^\delta$, so the model consistency term remains in the measurement domain.

In [ ]:
projectors = {}
fbp_solvers = {}
tpv_solvers = {}

for n_views in ANGLE_COUNTS:
    angle_key = str(n_views)
    K = operators.CTProjector(
        img_shape=(IMAGE_SIZE, IMAGE_SIZE),
        angles=np.linspace(0.0, np.pi, n_views),
        det_size=DETECTOR_SIZE,
        geometry=GEOMETRY,
    )
    projectors[angle_key] = K
    fbp_solvers[angle_key] = solvers.FBP(K)
    tpv_solvers[angle_key] = solvers.ChambollePockTpVUnconstrained(K)

print("Initialized operators for angles:", list(projectors.keys()))

## FBP as Initialization and Baseline

Filtered backprojection is not the variational method and is not used as an image-domain post-processing input. It is computed from the same sinogram for two controlled reasons: it provides a useful initial point $x_0$ for Chambolle-Pock, and it gives a classical baseline for the final tables and figures.

In [ ]:
def normalize_finite(image: torch.Tensor) -> torch.Tensor:
    image = torch.nan_to_num(image.float(), nan=0.0, posinf=0.0, neginf=0.0)
    if torch.isclose(image.max(), image.min()):
        return torch.zeros_like(image, dtype=torch.float32)
    return normalize(image).clamp(0.0, 1.0).to(torch.float32)


def prepare_image_slice(tensor: torch.Tensor, index: int) -> torch.Tensor:
    return torch.nan_to_num(
        tensor[index : index + 1].to(torch.float32),
        nan=0.0,
        posinf=1.0,
        neginf=0.0,
    ).clamp(0.0, 1.0).to(device)


def prepare_sinogram_slice(tensor: torch.Tensor, index: int) -> torch.Tensor:
    return torch.nan_to_num(
        tensor[index : index + 1].to(torch.float32),
        nan=0.0,
        posinf=0.0,
        neginf=0.0,
    ).to(device)


def run_fbp(angle_key: str, y_delta: torch.Tensor, x_true: torch.Tensor) -> torch.Tensor:
    x_fbp, _ = fbp_solvers[angle_key](
        y_delta,
        x_true=x_true,
        starting_point=y_delta,
    )
    return normalize_finite(x_fbp.detach().cpu()).to(device)


def run_tpv(angle_key: str, y_delta: torch.Tensor, x_true: torch.Tensor, x_fbp: torch.Tensor, params: dict) -> torch.Tensor:
    if not (0.1 < params["p"] < 0.5):
        raise ValueError(f"TpV parameter p must satisfy 0.1 < p < 0.5, got {params['p']}")

    x_sol, info = tpv_solvers[angle_key](
        y_delta,
        x_true=x_true,
        starting_point=x_fbp,
        lmbda=params["lmbda"],
        maxiter=params["maxiter"],
        tolf=params["tolf"],
        tolx=params["tolx"],
        p=params["p"],
        verbose=False,
    )
    return normalize_finite(x_sol.detach().cpu()).to(device)


def compute_metrics(reconstruction: torch.Tensor, target: torch.Tensor) -> dict:
    reconstruction_cpu = torch.clamp(reconstruction.detach().cpu(), 0.0, 1.0)
    target_cpu = torch.clamp(target.detach().cpu(), 0.0, 1.0)
    return {
        "psnr": float(metrics.PSNR(reconstruction_cpu, target_cpu)),
        "ssim": float(metrics.SSIM(reconstruction_cpu, target_cpu)),
    }


def absolute_error(reconstruction: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
    return torch.abs(
        torch.clamp(reconstruction.detach().cpu(), 0.0, 1.0)
        - torch.clamp(target.detach().cpu(), 0.0, 1.0)
    )


def center_crop(image: torch.Tensor, crop_size: int = 96) -> torch.Tensor:
    h, w = image.shape[-2:]
    top = (h - crop_size) // 2
    left = (w - crop_size) // 2
    return image[..., top : top + crop_size, left : left + crop_size]


def show_reconstruction_panel(x_true: torch.Tensor, x_fbp: torch.Tensor, x_tpv: torch.Tensor, title: str) -> None:
    x_true_cpu = torch.clamp(x_true.detach().cpu()[0, 0], 0.0, 1.0)
    x_fbp_cpu = torch.clamp(x_fbp.detach().cpu()[0, 0], 0.0, 1.0)
    x_tpv_cpu = torch.clamp(x_tpv.detach().cpu()[0, 0], 0.0, 1.0)
    fbp_error = absolute_error(x_fbp_cpu, x_true_cpu)
    tpv_error = absolute_error(x_tpv_cpu, x_true_cpu)

    images = [
        (x_true_cpu, "Ground Truth", "gray"),
        (x_fbp_cpu, "FBP", "gray"),
        (x_tpv_cpu, "TpV", "gray"),
        (fbp_error, "|FBP - GT|", "magma"),
        (tpv_error, "|TpV - GT|", "magma"),
        (center_crop(x_tpv_cpu), "TpV crop", "gray"),
    ]

    fig, axes = plt.subplots(2, 3, figsize=(10, 7), constrained_layout=True)
    fig.suptitle(title)
    for ax, (image, subtitle, cmap) in zip(axes.ravel(), images):
        ax.imshow(image, cmap=cmap, vmin=0.0, vmax=1.0 if cmap == "gray" else None)
        ax.set_title(subtitle)
        ax.axis("off")
    plt.show()

## Manual Parameter Calibration

The following cell is intentionally manual rather than an automatic grid search. A single combination of `lmbda`, `p`, and `maxiter` is tested on the central slice. The printed PSNR and SSIM, together with the visual panel, can be used to adjust the values before fixing the final parameters for the full volume.

In [ ]:
calibration_index = clean.shape[0] // 2

test_angle = "60"
test_params = {
    "lmbda": 0.01,
    "p": 0.35,
    "maxiter": 100,
    "tolf": 1e-5,
    "tolx": 1e-5,
}

x_true_cal = prepare_image_slice(clean, calibration_index)
y_delta_cal = prepare_sinogram_slice(sinograms[test_angle], calibration_index)

x_fbp_cal = run_fbp(test_angle, y_delta_cal, x_true_cal)
x_tpv_cal = run_tpv(test_angle, y_delta_cal, x_true_cal, x_fbp_cal, test_params)

fbp_cal_metrics = compute_metrics(x_fbp_cal, x_true_cal)
tpv_cal_metrics = compute_metrics(x_tpv_cal, x_true_cal)

print("Calibration slice:", calibration_index)
print("Angle:", test_angle)
print("Parameters:", test_params)
print(f"FBP  PSNR: {fbp_cal_metrics['psnr']:.4f} | SSIM: {fbp_cal_metrics['ssim']:.4f}")
print(f"TpV  PSNR: {tpv_cal_metrics['psnr']:.4f} | SSIM: {tpv_cal_metrics['ssim']:.4f}")

show_reconstruction_panel(
    x_true_cal,
    x_fbp_cal,
    x_tpv_cal,
    title=f"Manual calibration - {test_angle} views, slice {calibration_index}",
)

## Final Hard-Coded Parameters

After manual inspection, the final parameters are fixed in a dictionary indexed by the number of projection views. These values can be edited after using the calibration cell. The tolerances `tolf` and `tolx` are kept active to allow early stopping before `maxiter` when the primal-dual updates have stabilized.

In [ ]:
TPV_PARAMS = {
    "180": {"lmbda": 0.005, "p": 0.45, "maxiter": 100, "tolf": 1e-5, "tolx": 1e-5},
    "90": {"lmbda": 0.0075, "p": 0.40, "maxiter": 120, "tolf": 1e-5, "tolx": 1e-5},
    "60": {"lmbda": 0.010, "p": 0.35, "maxiter": 150, "tolf": 1e-5, "tolx": 1e-5},
    "45": {"lmbda": 0.015, "p": 0.30, "maxiter": 150, "tolf": 1e-5, "tolx": 1e-5},
}

for angle_key, params in TPV_PARAMS.items():
    if not (0.1 < params["p"] < 0.5):
        raise ValueError(f"Invalid p for {angle_key} views: {params['p']}")
    print(angle_key, params)

## Full-Volume Slice-by-Slice Reconstruction

The full test patient is reconstructed one slice at a time. For each slice and angle, the sinogram is moved to the selected device, FBP is computed from the sinogram, and the TpV solver is initialized with the FBP reconstruction. The final output mirrors the clean structure of notebook 01: one global patient dictionary containing all four angular configurations.

In [ ]:
reconstruction_output = {
    "clean": clean.detach().cpu().to(torch.float32),
    "reconstructions": {},
    "source_paths": source_paths,
    "metadata": {
        **metadata,
        "method": "ChambollePockTpVUnconstrained",
        "fbp_role": "starting_point_and_baseline",
        "device": str(device),
    },
}

for angle_key in [str(n_views) for n_views in ANGLE_COUNTS]:
    params = TPV_PARAMS[angle_key]
    fbp_slices = []
    tpv_slices = []

    print(f"Reconstructing {angle_key} views with parameters: {params}")
    for index in range(clean.shape[0]):
        x_true = prepare_image_slice(clean, index)
        y_delta = prepare_sinogram_slice(sinograms[angle_key], index)

        x_fbp = run_fbp(angle_key, y_delta, x_true)
        x_tpv = run_tpv(angle_key, y_delta, x_true, x_fbp, params)

        fbp_slices.append(x_fbp.detach().cpu())
        tpv_slices.append(x_tpv.detach().cpu())

        if (index + 1) % 25 == 0 or (index + 1) == clean.shape[0]:
            print(f"{angle_key} views: reconstructed {index + 1}/{clean.shape[0]} slices")

    reconstruction_output["reconstructions"][angle_key] = {
        "fbp": torch.cat(fbp_slices, dim=0).to(torch.float32),
        "tpv": torch.cat(tpv_slices, dim=0).to(torch.float32),
        "params": params,
    }

    del fbp_slices
    del tpv_slices
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

reconstruction_path = RECONSTRUCTION_DIR / "test_patient_reconstruction.pt"
torch.save(reconstruction_output, reconstruction_path)
print("Saved reconstruction file:", reconstruction_path)

## Quantitative Evaluation

PSNR and SSIM are computed for both FBP and TpV against the clean test volume. The table summarizes the behavior as the number of projections decreases: fewer views usually produce stronger streak artifacts in FBP, while TpV is expected to reduce geometric artifacts through sparse gradient regularization.

In [ ]:
metrics_rows = []

for angle_key in [str(n_views) for n_views in ANGLE_COUNTS]:
    entry = reconstruction_output["reconstructions"][angle_key]
    fbp_metrics = compute_metrics(entry["fbp"], clean)
    tpv_metrics = compute_metrics(entry["tpv"], clean)

    row = {
        "patient_id": patient_id,
        "angle_key": angle_key,
        "fbp_psnr": fbp_metrics["psnr"],
        "fbp_ssim": fbp_metrics["ssim"],
        "tpv_psnr": tpv_metrics["psnr"],
        "tpv_ssim": tpv_metrics["ssim"],
        "lmbda": TPV_PARAMS[angle_key]["lmbda"],
        "p": TPV_PARAMS[angle_key]["p"],
        "maxiter": TPV_PARAMS[angle_key]["maxiter"],
        "tolf": TPV_PARAMS[angle_key]["tolf"],
        "tolx": TPV_PARAMS[angle_key]["tolx"],
    }
    metrics_rows.append(row)
    print(
        f"{angle_key} views | "
        f"FBP PSNR {row['fbp_psnr']:.4f}, SSIM {row['fbp_ssim']:.4f} | "
        f"TpV PSNR {row['tpv_psnr']:.4f}, SSIM {row['tpv_ssim']:.4f}"
    )

metrics_path = METRICS_DIR / "tpv_metrics.csv"
with metrics_path.open("w", newline="", encoding="utf-8") as file:
    writer = csv.DictWriter(file, fieldnames=list(metrics_rows[0].keys()))
    writer.writeheader()
    writer.writerows(metrics_rows)

print("Saved metrics:", metrics_path)

## Visual Comparison

The visual panels compare the clean reference, the FBP baseline, the TpV reconstruction, and the absolute error maps. A central crop is included to inspect edges and local geometric structure, where sparse-view streaking is typically most visible.

In [ ]:
visual_index = calibration_index

for angle_key in [str(n_views) for n_views in ANGLE_COUNTS]:
    entry = reconstruction_output["reconstructions"][angle_key]

    x_true = clean[visual_index : visual_index + 1]
    x_fbp = entry["fbp"][visual_index : visual_index + 1]
    x_tpv = entry["tpv"][visual_index : visual_index + 1]

    x_true_img = torch.clamp(x_true[0, 0], 0.0, 1.0)
    x_fbp_img = torch.clamp(x_fbp[0, 0], 0.0, 1.0)
    x_tpv_img = torch.clamp(x_tpv[0, 0], 0.0, 1.0)
    fbp_error = absolute_error(x_fbp_img, x_true_img)
    tpv_error = absolute_error(x_tpv_img, x_true_img)

    panels = [
        (x_true_img, "Ground Truth", "gray"),
        (x_fbp_img, "FBP", "gray"),
        (x_tpv_img, "TpV", "gray"),
        (fbp_error, "|FBP - GT|", "magma"),
        (tpv_error, "|TpV - GT|", "magma"),
        (center_crop(x_tpv_img), "TpV crop", "gray"),
    ]

    fig, axes = plt.subplots(2, 3, figsize=(10, 7), constrained_layout=True)
    fig.suptitle(f"TpV reconstruction - {angle_key} views - slice {visual_index}")

    for ax, (image, title, cmap) in zip(axes.ravel(), panels):
        ax.imshow(image, cmap=cmap, vmin=0.0, vmax=1.0 if cmap == "gray" else None)
        ax.set_title(title)
        ax.axis("off")

    figure_path = FIGURES_DIR / f"tpv_comparison_{angle_key}.png"
    fig.savefig(figure_path, dpi=150)
    plt.show()
    plt.close(fig)
    print("Saved figure:", figure_path)